In [ ]:
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# core libs for eda and audio
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

BASE   = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
ESC50  = os.path.join(BASE, 'ESC-50-master')
SR     = 22050

train_df = pd.read_csv(os.path.join(BASE, 'train.csv'))
test_df  = pd.read_csv(os.path.join(BASE, 'test.csv'))
sub_df   = pd.read_csv(os.path.join(BASE, 'sample_submission.csv'))

print(train_df.shape, test_df.shape)
print(train_df.head())

In [ ]:
# quick look at class balance
plt.figure(figsize=(10, 4))
vc = train_df['genre'].value_counts()
sns.barplot(x=vc.index, y=vc.values, palette='viridis')
plt.title('Class Distribution')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

print(vc)
print(f'\nImbalance ratio (max/min): {vc.max()/vc.min():.2f}')

In [ ]:
# audio length distribution across classes
filename_col = [c for c in train_df.columns if 'file' in c.lower() or 'name' in c.lower() or 'path' in c.lower()][0]

durations = []
for _, row in train_df.iterrows():
    fp = os.path.join(BASE, row[filename_col])
    try:
        dur = librosa.get_duration(path=fp)
    except:
        dur = np.nan
    durations.append(dur)

train_df['duration'] = durations

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
train_df['duration'].hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Audio Duration Distribution')
axes[0].set_xlabel('Seconds')
train_df.boxplot(column='duration', by='genre', ax=axes[1], rot=30)
axes[1].set_title('Duration by Genre')
plt.suptitle('')
plt.tight_layout()
plt.show()

print(train_df['duration'].describe())

In [ ]:
# checking silence/gaps inside clips
def silence_ratio(fp, top_db=30):
    try:
        y, sr = librosa.load(fp, sr=SR, mono=True)
        non_silent = librosa.effects.split(y, top_db=top_db)
        voiced_samples = sum(e - s for s, e in non_silent)
        return 1 - (voiced_samples / len(y))
    except:
        return np.nan

sample_paths = train_df[filename_col].sample(200, random_state=42).apply(lambda f: os.path.join(BASE, f))
silence_ratios = sample_paths.apply(silence_ratio)

plt.figure(figsize=(8, 4))
sns.histplot(silence_ratios.dropna(), bins=30, kde=True, color='coral')
plt.title('Silence Ratio in 200-Sample Subset')
plt.xlabel('Fraction of Silent Samples')
plt.tight_layout()
plt.show()

print(f'Mean silence ratio: {silence_ratios.mean():.3f}')
print(f'Files >50% silence : {(silence_ratios > 0.5).sum()}')

In [ ]:
# exploring ESC-50 noise samples that are mixed in
esc_meta = pd.read_csv(os.path.join(ESC50, 'meta', 'esc50.csv'))
print(esc_meta.shape)
print(esc_meta['category'].value_counts().head(15))

plt.figure(figsize=(12, 4))
vc_esc = esc_meta['category'].value_counts().head(20)
sns.barplot(x=vc_esc.index, y=vc_esc.values, palette='magma')
plt.title('Top 20 ESC-50 Noise Categories')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# waveform + spectrogram for one sample per genre
genres = train_df['genre'].unique()
fig, axes = plt.subplots(len(genres), 2, figsize=(14, len(genres) * 2.5))

for i, genre in enumerate(genres):
    fp = os.path.join(BASE, train_df[train_df['genre'] == genre].iloc[0][filename_col])
    y, sr = librosa.load(fp, sr=SR, duration=10)
    mel   = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=64)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    axes[i, 0].plot(y, linewidth=0.5, color='steelblue')
    axes[i, 0].set_title(f'{genre} - waveform')
    axes[i, 0].axis('off')

    librosa.display.specshow(mel_db, sr=sr, ax=axes[i, 1], cmap='magma')
    axes[i, 1].set_title(f'{genre} - mel spectrogram')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# random baseline — most frequent class for every test sample
most_common_genre = train_df['genre'].value_counts().idxmax()
sub_df['genre']   = most_common_genre
sub_df.to_csv('/kaggle/working/submission_baseline.csv', index=False)
print(f'Baseline class used: {most_common_genre}')
print(sub_df.head())